# <center> Embedding

## Sources

https://github.com/UKPLab/sentence-transformers<br>

**Semantic Search**<br>
https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html<br>
https://www.sbert.net/examples/sparse_encoder/applications/semantic_search/README.html<br>


## Imports

In [197]:
%reset

In [1]:
import os
import re 

import numpy as np
import pandas as pd

# import faiss
import spacy
from spacy.tokenizer import Tokenizer

from sentence_transformers import SentenceTransformer
# from transformers import AutoModel, AutoTokenizer
# from transformers import CamembertTokenizer

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import PythonCodeTextSplitter
# from langchain.text_splitter import SpacyTextSplitter
# from langchain.text_splitter import TokenTextSplitter
# from langchain.text_splitter import NLTKTextSplitter

import faiss
#%pip install faiss-cpu

#pd.set_option('display.max_columns', 5)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 20)

nlp = spacy.load("fr_core_news_md")

c:\Users\S589669\Documents\DEV\ABP-Doctheque\env_doctheque\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Documents

In [2]:
filename = "CSV/documents_cleaned.csv"
df = pd.read_csv(filename, index_col = [0])
titles = df["Titles_Cleaned"].values
documents = df["Documents_Cleaned"].values
print(type(titles), type(documents))
df

<class 'numpy.ndarray'> <class 'numpy.ndarray'>


,Ext,Titles,Titles_Cleaned,Documents_Cleaned
0,pptx,02 - Process réclamations,02 - process réclamations,gestion des reclamations organisation car animation et coordination journalière avec répartition...
1,pptx,AAV - Dérogations D08 - D10,aav - dérogations d08 - d10,aav - dérogations d08-d10 délégation équipe car délégation animateur analyste ru à indiquer dans...
2,pptx,Analyser un impayé,analyser un impayé,faire la cf04 avec la référence bancaire pour vérifier si paiements en retard on régularise l’en...
3,docx,Automatisation des impayés,automatisation des impayés,automatisation des impayes procédure à utiliser après analyse de la cf01 et cf04 et cf02 si néce...
4,docx,calcul remboursement,calcul remboursement,v/ 3.60 - 1.01 liste des reglements clients 822s506813 client 82200000081114473 sce societe civi...
5,docx,DEDOUBLONNAGE PACIFICA,dedoublonnage pacifica,procédure historique des modifications table des matières contenu description du processus conte...
6,pdf,DEROGATIONS A VALIDER PP07,derogations a valider pp07,procédure dérogations pp07 à valider rôle nom prénom rédaction miot francois contribution valida...
7,pdf,DEROGATIONS MISE à JOUR 04.2025,derogations mise à jour 04.2025,procédure procédure dérogations pp07 à valider rôle nom prénom rédaction guyon amélie contributi...
8,pdf,Dérogations PP07 - L'Essentiel,dérogations pp07 - l'essentiel,derogations a valider via la pp07 préambule - un assureur n’a pas à motiver un refus d’entrée du...
9,docx,FICHE DE RENSEIGNEMENT REGUL IMPAYE PACIFICA,fiche de renseignement regul impaye pacifica,fiche de renseignement : regularisation impayes pacifica client concerne nom : . prenom : . réf ...


## Tokenization

**Two different ways to tokenize**

tokenizer() AND nlp()<br>

nlp() recognizes slashes between words while tokenizer() does not.

In [7]:
n_doc = 0
document = df['Documents_Cleaned'][n_doc]

# Create a tokenizer with nlp.vocab
tokenizer = Tokenizer(nlp.vocab)
tokens = tokenizer(document)

# Default tokenizer
doc = nlp(document)
print(type(tokens), type(doc))

for i, j in zip(tokens[:100], doc[:100]):
    print(i, j)

<class 'spacy.tokens.doc.Doc'> <class 'spacy.tokens.doc.Doc'>
gestion gestion
des des
reclamations reclamations
organisation organisation
car car
animation animation
et et
coordination coordination
journalière journalière
avec avec
répartition répartition
des des
réclamations réclamations
une une
équipe équipe
polyvalente polyvalente
de de
18 18
conseillers conseillers
qui qui
traite traite
les les
réclamations réclamations
des des
journées journées
dédiées dédiées
et et
un un
suivi suivi
personnel personnel
des des
réclamations réclamations
prises prises
en en
charge charge
veille veille
permanente permanente
sur sur
les les
réclamations réclamations
en en
cours/suivi cours
par /
binôme suivi
si par
absence binôme
une si
boîte absence
mail une
dédiée boîte
aux mail
réclamations dédiée
sommaire aux
contexte réclamations
objectifs sommaire
detail contexte
de objectifs
la detail
procedure de
demandes la
de procedure
renseignements demandes
assistance de
car renseignements
conversation as

In [3]:
def remove_punct_digit(text):
    doc = nlp(text)
    return [token for token in doc if not (token.is_punct or token.is_digit)]

def remove_stop_words_punct(text):
    doc = nlp(text)
    return [token for token in doc if not (token.is_stop or token.is_punct)]

df_tokens = df.copy()
df_tokens['Titles_Token'] = df_tokens['Titles_Cleaned'].apply(remove_punct_digit)
df_tokens['Documents_Token'] = df_tokens['Documents_Cleaned'].apply(remove_punct_digit)
# df_tokens['Titles_Token'] = df_tokens['Titles_Cleaned'].apply(remove_stop_words_punct)
# df_tokens['Documents_Token'] = df_tokens['Documents_Cleaned'].apply(remove_stop_words_punct)
df_tokens.drop(columns = ['Titles_Cleaned', 'Documents_Cleaned'], inplace = True)
df_tokens

,Ext,Titles,Titles_Token,Documents_Token
0,pptx,02 - Process réclamations,"[process, réclamations]","[gestion, des, reclamations, organisation, car, animation, et, coordination, journalière, avec, ..."
1,pptx,AAV - Dérogations D08 - D10,"[aav, dérogations, d08, d10]","[aav, dérogations, d08-d10, délégation, équipe, car, délégation, animateur, analyste, ru, à, ind..."
2,pptx,Analyser un impayé,"[analyser, un, impayé]","[faire, la, cf04, avec, la, référence, bancaire, pour, vérifier, si, paiements, en, retard, on, ..."
3,docx,Automatisation des impayés,"[automatisation, des, impayés]","[automatisation, des, impayes, procédure, à, utiliser, après, analyse, de, la, cf01, et, cf04, e..."
4,docx,calcul remboursement,"[calcul, remboursement]","[v/, 3.60, 1.01, liste, des, reglements, clients, 822s506813, client, sce, societe, civile, d', ..."
5,docx,DEDOUBLONNAGE PACIFICA,"[dedoublonnage, pacifica]","[procédure, historique, des, modifications, table, des, matières, contenu, description, du, proc..."
6,pdf,DEROGATIONS A VALIDER PP07,"[derogations, a, valider, pp07]","[procédure, dérogations, pp07, à, valider, rôle, nom, prénom, rédaction, miot, francois, contrib..."
7,pdf,DEROGATIONS MISE à JOUR 04.2025,"[derogations, mise, à, jour, 04.2025]","[procédure, procédure, dérogations, pp07, à, valider, rôle, nom, prénom, rédaction, guyon, améli..."
8,pdf,Dérogations PP07 - L'Essentiel,"[dérogations, pp07, l', essentiel]","[derogations, a, valider, via, la, pp07, préambule, un, assureur, n’, a, pas, à, motiver, un, re..."
9,docx,FICHE DE RENSEIGNEMENT REGUL IMPAYE PACIFICA,"[fiche, de, renseignement, regul, impaye, pacifica]","[fiche, de, renseignement, , regularisation, impayes, pacifica, client, concerne, nom, , preno..."


**Rebuild document from tokens**

NOT WORKING

In [181]:
n_doc = 0
document = documents[n_doc]
tokens = nlp(document)

document_ = ' '.join([token.text for token in tokens])
print(document)
print(document_)

gestion des reclamations organisation car animation et coordination journalière avec répartition des réclamations une équipe polyvalente de 18 conseillers qui traite les réclamations des journées dédiées et un suivi personnel des réclamations prises en charge veille permanente sur les réclamations en cours/suivi par binôme si absence une boîte mail dédiée aux réclamations sommaire contexte objectifs detail de la procedure demandes de renseignements assistance car conversation à trois réclamations analyse décision /pec regularisation rédaction de la fiche pec envoi à usp pour traitement annexes organigramme écoute client par cr annuaire ugs coordonnées utiles contexte car est en charge du traitement des demandes de renseignements et des réclamations clients sur le périmètre iard part et pro et sur la prévoyance part et pro objectifs traitement en amont des demandes de renseignements en soutien des conseillers traitement des réclamations saisies en ave by caesar en recherchant la satisfa

## Chunking

Several technics to chunk documents
- tokens chunking
- semantic chunking
...

In [4]:
# 1. Configuration
class Config:
    MODEL_NAME = 'all-MiniLM-L6-v2'  # Modèle BERT léger
    CHUNK_SIZE = 256                 # Taille des chunks en tokens
    CHUNK_OVERLAP = 48               # Chevauchement entre chunks
    INDEX_FILE = "document_index.faiss"
    METADATA_FILE = "metadata.json"
    DOCUMENTS_DIR = "cleaned_docs"   # Dossier contenant vos documents nettoyés

# 3. Découpeur de texte
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=Config.CHUNK_SIZE,
    chunk_overlap=Config.CHUNK_OVERLAP,
    length_function=len
)

@np.vectorize
def light_cleaning(document):
    ''' light cleaning for embedding '''
    document = document.replace('«', ' ').replace('»', ' ').replace('>', ' ').replace('<', ' ')
    document = document.replace('-', ' ').replace('+', ' ').replace('=', ' ').replace("n°", ' ')
    document = document.replace(':', ' ').replace(',', ' ').replace('.', ' ')
    document = document.replace('€', ' ')
    # Remove non-breaking spaces
    document = document.replace("\xa0", ' ')
    document = re.sub(r" +", ' ', document).strip()
    return document

# Cleaning the documents
cleaned_documents = light_cleaning(documents)
print(type(cleaned_documents), type(cleaned_documents.item(0)))

n_doc = 3
document = cleaned_documents[n_doc]
print(titles[n_doc])
print(document)

chunks = text_splitter.split_text(document.item(0))
print("\nCHUNKS:")
for chunk in chunks:
    print(" - ", chunk)

<class 'numpy.ndarray'> <class 'str'>
automatisation des impayés
automatisation des impayes procédure à utiliser après analyse de la cf01 et cf04 et cf02 si nécessaire utilisation se connecter au pvm de la cr concernée par l’impayé à régulariser compléter le fichier trame impayés tc01 le fichier se trouve sous s \dab abp car\equip\02 gerer les contrats de la souscription à la résiliation impayés \automatisation des impayes\trame impayes tc01 attention dans les pvm autre que la 22 le fichier se trouve sous s \dab abp car\equip\02 gerer les contrats de la souscription à la résiliation impayés \automatisation des impayes\trame impayes tc01 a noter que l’intêret de satambo est de remplir le fichier excel avec plusieurs impayés de clients différents et de toutes cr le but étant de traiter en masse attention fichier a remplir en majuscules ne compléter que l’onglet impayés colonne a numéro de compte client colonne b nom et prénom du client colonne c montant impayé avec frais du contrat à rég

**Benchmark vectorization**

In [237]:
%timeit light_cleaning(cleaned_documents)
%timeit [light_cleaning(doc) for doc in cleaned_documents]
%timeit map(light_cleaning, cleaned_documents)

3.11 ms ± 59.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.33 ms ± 76.5 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
49.8 ns ± 0.616 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


**Test vectorisation**

In [5]:
for chunk in chunks:
    print("nb tokens:", len(nlp(chunk)))

nb tokens: 44
nb tokens: 42
nb tokens: 42
nb tokens: 45
nb tokens: 46
nb tokens: 49
nb tokens: 46
nb tokens: 40
nb tokens: 40
nb tokens: 44
nb tokens: 47
nb tokens: 46
nb tokens: 39
nb tokens: 41
nb tokens: 39
nb tokens: 42
nb tokens: 45
nb tokens: 40
nb tokens: 49
nb tokens: 46
nb tokens: 46
nb tokens: 45
nb tokens: 40
nb tokens: 41
nb tokens: 38
nb tokens: 42
nb tokens: 40
nb tokens: 41
nb tokens: 39
nb tokens: 36
nb tokens: 42
nb tokens: 43
nb tokens: 42
nb tokens: 47
nb tokens: 14


## Sentence-camembert-large

In [6]:
# https://datacorner.fr/document-chunking/
from langchain_huggingface import HuggingFaceEmbeddings

# hf_embedding = HuggingFaceEmbeddings(model)

**Locally load the model**

In [7]:
model = SentenceTransformer(r"C:\Users\S589669\Documents\DEV\ABP-Doctheque\sentence-camembert-large", local_files_only=True)
model.encode("Hello")

array([ 0.29252246,  0.02982062, -0.10471618, ...,  0.22709577,
        0.05306111, -0.58324724], shape=(1024,), dtype=float32)

In [ ]:
# Documents are too long (need to be chunked)
n_index = 0
print("max_seq_length", model.max_seq_length)
model.max_seq_length = 1024
print("max_seq_length", model.max_seq_length)
# document = ["Procédure Historique des modifications Table des matières Contenu Description du processus Contexte Action à mener lorsque le partenaire est déjà connu dans la base sous une autre référence ce qui", 'Procédure Rajout codes promo si oubli conseiller Rôle Nom Prénom Rédaction HAMON Joel Contribution XXXXX Validation XXXXX Historique des modifications Validation Version Date Rôle']

print("len documents", len(documents[n_index]))
corpus_embeddings = model.encode(documents[n_index], convert_to_numpy=True)
print("Shape: ", corpus_embeddings.shape)
corpus_embeddings

**Documents chunking**

In [ ]:
chunks = []
# Indexes_documents sert de mapping entre les chunks et les documents d'origine
indexes_documents = []

for i, document in enumerate(cleaned_documents):
    chunks_ = text_splitter.split_text(document)
    chunks.extend(chunks_)
    indexes_documents.extend([i] * len(chunks_))

print('Documents length:')
for i, document in enumerate(cleaned_documents):
    print(f"nb tokens: {len(nlp(document.item(0)))}\t nb chunks: {indexes_documents.count(i)}")
print(f"nb total chunks: {len(indexes_documents)}")

print("\nCHUNKS:")
for i, chunk in enumerate(chunks):
    print(f"{indexes_documents[i]} - {chunk}")

Documents length:
nb tokens: 1170	 nb chunks: 36
nb tokens: 23	 nb chunks: 1
nb tokens: 180	 nb chunks: 5
nb tokens: 1198	 nb chunks: 35
nb tokens: 135	 nb chunks: 4
nb tokens: 159	 nb chunks: 5
nb tokens: 2746	 nb chunks: 71
nb tokens: 2427	 nb chunks: 65
nb tokens: 966	 nb chunks: 26
nb tokens: 43	 nb chunks: 2
nb tokens: 793	 nb chunks: 22
nb tokens: 616	 nb chunks: 18
nb tokens: 1062	 nb chunks: 29
nb tokens: 378	 nb chunks: 10
nb tokens: 819	 nb chunks: 24
nb tokens: 517	 nb chunks: 16
nb tokens: 92	 nb chunks: 3
nb tokens: 224	 nb chunks: 6
nb total chunks: 378

CHUNKS:
0 - gestion des reclamations organisation car animation et coordination journalière avec répartition des réclamations une équipe polyvalente de 18 conseillers qui traite les réclamations des journées dédiées et un suivi personnel des réclamations prises en
0 - un suivi personnel des réclamations prises en charge veille permanente sur les réclamations en cours/suivi par binôme si absence une boîte mail dédiée aux r

**Chunks embedding**

In [11]:
chunks_embedding = model.encode(chunks, convert_to_numpy=True)
print("Shape: ", chunks_embedding.shape)
chunks_embedding

Shape:  (378, 1024)


array([[ 0.5831421 ,  0.01211701, -0.22931232, ...,  0.0956009 ,
         0.28513888, -0.08372404],
       [ 0.6605553 , -0.13593258, -0.17265643, ..., -0.01612344,
         0.11497779,  0.08793126],
       [ 0.5812895 , -0.2690465 ,  0.30170736, ...,  0.12867296,
         0.48447734, -0.4153752 ],
       ...,
       [ 0.4819529 , -0.43881786,  0.03134119, ..., -0.1581173 ,
        -0.04589377, -0.0895991 ],
       [ 0.33758026, -0.75530684,  0.35650203, ..., -0.3526304 ,
         0.12827727, -0.05234856],
       [ 0.847971  , -0.2267644 ,  0.19055344, ...,  0.10415933,
         0.36788565, -0.19434743]], shape=(378, 1024), dtype=float32)

**Query embedding**

In [13]:
# Query sentences:
queries = [
    "dérogations",
    "dérogation",
    "derogation",
    "derogations",
    "animateur analyste.",
    "dérogations hors norme.",
    "règlement des impayés",
    "impayés",
    "impayé",
    "multi détention",
    "gestion des doublons de dossier",
    "résiliations"
]

# queries_cleaned = [light_cleaning(query) for query in queries]
queries_cleaned = light_cleaning(queries)
queries_embedding = model.encode(queries_cleaned, convert_to_numpy=True)
top_k = min(5, len(documents))

for i, query_embedding in enumerate(queries_embedding):
    print(queries[i])
    similarities = model.similarity(queries_embedding[i], chunks_embedding)[0].numpy()
    # print(similarities.shape)
    index_scores = similarities.argsort()
    # print(index_scores)

    counter = 0
    indexes_result = []
    for j in index_scores[::-1]:
        if indexes_documents[j] in indexes_result:
            continue
        print(f" - {titles[indexes_documents[j]]: <50}: {similarities[j]:.4f}")
        indexes_result.append(indexes_documents[j])
        if len(indexes_result) == top_k:
            break

dérogations
 - derogations a valider pp07                        : 0.4921
 - process hors norme 004                            : 0.4719
 - derogations mise à jour 04.2025                   : 0.4531
 - reclamations                                      : 0.3952
 - dérogations pp07 - l'essentiel                    : 0.3931
dérogation
 - derogations a valider pp07                        : 0.4117
 - process hors norme 004                            : 0.3675
 - derogations mise à jour 04.2025                   : 0.3546
 - dérogations pp07 - l'essentiel                    : 0.3362
 - reedition des contrats suite dysfonctionnement impression: 0.3324
derogation
 - derogations a valider pp07                        : 0.4886
 - reclamations                                      : 0.3742
 - process hors norme 004                            : 0.3641
 - derogations mise à jour 04.2025                   : 0.3380
 - 02 - process réclamations                         : 0.3143
derogations
 - derogations a 

## all-minilm-l6-v2

**Locally load the model (NOT WORKING)**

In [ ]:
# model = SentenceTransformer(r"C:\Users\S589669\Documents\DEV\ABP-Doctheque\all-minilm-l6-v2-tensorflow2-default-v1",
#                             local_files_only=True, word_embedding_dimension = 512)

model = SentenceTransformer(r"C:\Users\S589669\Documents\DEV\ABP-Doctheque\all-minilm-l6-v2-tensorflow2-default-v1", local_files_only=True)
model.encode("Hello")

## Indexation with FAISS